In [1]:
# 라이브러리 호출
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings
import platform

warnings.filterwarnings('ignore')

# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False

In [2]:
# 데이터 프레임을 출력할 때, 행과 컬럼이 모두 생략되지 않도록 하는 코드
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

df = pd.read_csv("DieCasting_Quality_Raw_Data.csv", header=[0,1])
print("df shape:", df.shape)
df.head(20)

df shape: (7535, 57)


Process                                                                   \
        id Product_Type Shot Velocity_1 Velocity_2 Velocity_3 High_Velocity   
0        1            1    1      0.144      0.170      0.188         2.134   
1     1002            1    2      0.144      0.170      0.182         2.124   
2     2003            1    3      0.144      0.170      0.182         2.116   
3     3004            1    4      0.144      0.170      0.182         2.137   
4     4005            1    5      0.144      0.172      0.176         2.111   
5     5006            1    6      0.144      0.172      0.176         2.111   
6     6008            1    8      0.141      0.168      0.178         2.159   
7     7009            1    9      0.142      0.172      0.176         2.114   
8     8010            1   10      0.142      0.172      0.176         2.114   
9     9011            1   11      0.144      0.168      0.184         2.145   
10   10012            1   12      0.144      0.168      0.184         2.145   
11   11013            1   13      0.144      0.166      0.176         2.120   
12   12014            1   14      0.144      0.170      0.176         2.112   
13   13015            1   15      0.140      0.168      0.184         2.154   
14   14016            1   16      0.142      0.168      0.178         2.130   
15   15017            1   17      0.142      0.168      0.176         2.128   
16   16018            1   18      0.142      0.168      0.176         2.142   
17   17019            1   19      0.142      0.168      0.182         2.114   
18   18020            1   20      0.142      0.166      0.182         2.131   
19   19021            1   21      0.142      0.170      0.185         2.147   

                                                                         \
   Cylinder_Pressure Rapid_Rise_Time Biscuit_Thickness  Clamping_Force    
0                214           0.008                 10             258   
1                217           0.008                 11             257   
2                214           0.008                 11             257   
3                217           0.008                 11             257   
4                217           0.008                 12             257   
5                217           0.008                 12             257   
6                214           0.008                 11             257   
7                217           0.008                 12             257   
8                217           0.008                 12             257   
9                217           0.008                 11             257   
10               217           0.008                 11             257   
11               217           0.008                 11             257   
12               214           0.010                 12             257   
13               217           0.008                 11             257   
14               214           0.007                 11             257   
15               217           0.008                 11             257   
16               214           0.008                 11             255   
17               217           0.009                 11             257   
18               214           0.008                 11             257   
19               217           0.008                 11             255   

                                                                            \
   Cycle_Time  Pressure_Rise_Time Casting_Pressure Spray_Time Spray_1_Time   
0        20.7               0.044             1037        7.8          0.7   
1        20.7               0.044             1052        7.8          0.7   
2        20.8               0.041             1037        7.8          0.7   
3        20.7               0.043             1051        7.8          0.7   
4        20.7               0.042             1052        7.8          0.7   
5        20.7               0.042             1052        7.8      

In [3]:
# =========================
# 2) 메인 데이터 컬럼 구조 확인 (Process / Sensor / Defects)
# =========================
# MultiIndex 컬럼의 0레벨(최상위)의 컬럼들만 잘라서 가져옴
process_df = df.xs("Process", axis=1, level=0)
sensor_df  = df.xs("Sensor",  axis=1, level=0)
defects_df = df.xs("Defects", axis=1, level=0)

process_df.columns = process_df.columns.astype(str).str.strip()
sensor_df.columns  = sensor_df.columns.astype(str).str.strip()
defects_df.columns = defects_df.columns.astype(str).str.strip()

# Defects 값을 0/1로 통일

Defects관련 컬럼을 불량 유형별로 정리하고, target 설정 -> Defects 관련 컬럼은 모두 삭제\
타겟(Defects) 하나가 어떤 데이터에 매칭되어 있는건지 관계를 확실히 확인

In [4]:
# 0보다 크면 불량(1)로 바꾸기
defects_df = defects_df.apply(pd.to_numeric, errors="coerce").fillna(0)
defects_df = (defects_df > 0).astype(int)
# defects_df = (defects_df > 0).astype(int) # 불량유형 지정 안됨

In [5]:
col_uniques = defects_df.apply(lambda s: sorted(s.dropna().unique()))
col_uniques

Short_Shot_1       [0, 1]
Bubble_1           [0, 1]
Exfoliation_1      [0, 1]
Blow_Hole_1        [0, 1]
Stain_1            [0, 1]
Dent_1             [0, 1]
Deformation_1      [0, 1]
Contamination_1    [0, 1]
Impurity_1         [0, 1]
Crack_1            [0, 1]
Scratch_1          [0, 1]
Buring_Mark_1      [0, 1]
Inclusions_1          [0]
Short_Shot_2       [0, 1]
Bubble_2           [0, 1]
Exfoliation_2      [0, 1]
Blow_Hole_2        [0, 1]
Stain_2               [0]
Dent_2             [0, 1]
Deformation_2      [0, 1]
Contamination_2    [0, 1]
Impurity_2         [0, 1]
Crack_2            [0, 1]
Scratch_2             [0]
Buring_Mark_2         [0]
Inclusions_2       [0, 1]
dtype: object

## Cavity 1/2 통합

In [6]:
# 원본 보관
defects_df_clean = defects_df.copy()

# cavity 1/2 컬럼 분리
c1 = defects_df_clean[[c for c in defects_df_clean.columns if c.endswith("_1")]].copy()
c2 = defects_df_clean[[c for c in defects_df_clean.columns if c.endswith("_2")]].copy()

# 컬럼명 통일: Short_Shot_1 -> Short_Shot
c1.columns = [c.replace("_1", "") for c in c1.columns]
c2.columns = [c.replace("_2", "") for c in c2.columns]

# 제대로 분리되었는지 확인
print("c1 shape:", c1.shape)
print("c2 shape:", c2.shape)
print("c1, c2 컬럼이 동일한가?", set(c1.columns) == set(c2.columns))

c1 shape: (7535, 13)
c2 shape: (7535, 13)
c1, c2 컬럼이 동일한가? True


In [7]:
# OR 방식으로 통합: 둘 중 하나라도 1이면 1
defects_merged = ((c1 + c2) > 0).astype(int)   # (c1 | c2) 도 가능
defects_df_clean = defects_merged

print("통합 전 defects_df shape:", defects_df.shape)
print("통합 후 defects_df_clean shape:", defects_df_clean.shape)
print("값 종류:", np.unique(defects_df_clean.to_numpy()))
defects_df_clean.head(20)

통합 전 defects_df shape: (7535, 26)
통합 후 defects_df_clean shape: (7535, 13)
값 종류: [0 1]


,Short_Shot,Bubble,Exfoliation,Blow_Hole,Stain,Dent,Deformation,Contamination,Impurity,Crack,Scratch,Buring_Mark,Inclusions
0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,1,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,1,0,0,0,0,0,0


In [8]:
# 전체 셀 기준(모든 defect 칸을 다 펼쳐서 0/1/2 비율)
flat = defects_df_clean.to_numpy().ravel()
dist_all = pd.Series(flat).value_counts(normalize=True).reindex([0,1,2], fill_value=0)
print("=== 전체 셀 기준 0/1/2 비율 ===")
print((dist_all * 100).round(2).astype(str) + "%")

=== 전체 셀 기준 0/1/2 비율 ===
0    98.16%
1     1.84%
2      0.0%
Name: proportion, dtype: object


## 각 불량 유형별로 묶음 (표면, 구조, 이물질 등)

1. Short_Shot: 미성형(충전 부족) 불량 — 용탕이 금형 끝까지 다 채워지지 않아 모서리/얇은 부위가 덜 찍히거나 일부가 비는 현상.
2. Bubble: 기포 불량 — 주조 중 갇힌 가스/공기가 내부나 표면에 기포로 남아 강도 저하·누설/외관 불량을 유발.
3. Exfoliation: 박리(층 분리/벗겨짐) 불량 — 표면층이 뜨거나 층이 분리되어 벗겨지는 형태로, 표면 품질·내구성을 크게 떨어뜨림.
4. Blow_Hole: 블로홀/기공 불량 — 응고 중 가스/수축으로 생긴 비교적 큰 공극(구멍)으로, 단면 기공·핀홀처럼 나타남.
5. Stain: 얼룩 불량 — 산화막/윤활제/오염 영향으로 표면 색이 불균일하게 남아 얼룩처럼 보이는 외관 불량.
6. Dent: 찍힘/눌림 불량 — 취출·이송·적재 과정에서 충격/압력으로 표면이 눌리거나 자국이 생긴 외관/치수 불량.
7. Scratch: 스크래치 불량 — 금형, 이송 설비, 공구, 적재 간 마찰로 표면에 선형 흠집이 생기는 외관 불량.
8. Buring_Mark: 번마크/그을림 불량 — 고온 가스/산화 영향으로 표면이 그을리거나 변색되는 현상(검게 타거나 갈변).
9. Deformation: 변형 불량 — 응고 후 취출·냉각·후공정 중 응력/열로 휘어짐·뒤틀림이 생겨 형상/치수가 틀어짐.
10. Crack: 균열 불량 — 응고수축·열응력·과도한 취출력 등으로 미세~육안 균열이 발생해 파손 위험이 커짐.
11. Contamination: 오염(이물) 불량 — 분진/오일/칩 등 외부 이물이 표면에 붙거나 끼어 도장·외관·조립성 문제를 만듦.
12. Impurity: 불순물 불량 — 용탕 내 불순 성분/산화물 등이 섞여 품질이 저하되고 표면 결함·취성 증가로 이어짐.
13. Inclusions: 개재물(내부 이물) 불량 — 산화막/슬래그 등 비금속물이 내부에 갇혀 단면에 점/덩어리로 보이며 강도를 약화.

- 표면 불량(surface_defect) : 육안으로 확인 가능하지만, 금속의 분리나 갈라짐은 없는 불량 (Stain, Dent, Scratch, Burning_Mark)
- 구조 불량(structural_defect): 육안으로 금속의 분리·갈라짐이 보이거나, 제품의 강도·기능에 직접 영향을 줄 수 있는 불량 (Short_Shot, Bubble, Blow_Hole, Deformation, Crack, Exfoliation)
- 이물질 포함 불량(contamination_defect): 원래 포함되면 안 되는 외부 물질이 들어간 불량 (Contamination, Impurity, Inclusions)

In [9]:
# 1) 그룹 정의
# 그룹별 OR(하나라도 1이면 1)로 합치기
group_df = pd.DataFrame(index=defects_df_clean.index)

group_df["Surface_Defect"] = ( # 표면 불량
    defects_df_clean.reindex(columns=["Stain", "Dent", "Scratch", "Burning_Mark"], fill_value=0).max(axis=1)
)

group_df["Structural_Defect"] = ( # 구조 불량
    defects_df_clean.reindex(columns=["Short_Shot", "Bubble", "Blow_Hole", "Deformation", "Crack", "Exfoliation"], fill_value=0).max(axis=1)
)

group_df["Contamination_Defect"] = ( # 이물질 포함 불량
    defects_df_clean
    .reindex(columns=["contamination", "impurity", "inclusions"], fill_value=0)
    .max(axis=1)
)

print("group_df shape:", group_df.shape)
print("값 종류:", np.unique(group_df.to_numpy()))
display((group_df.mean() * 100).round(2).sort_values(ascending=False))
display(group_df.sum().sort_values(ascending=False))
group_df.info()

group_df shape: (7535, 3)
값 종류: [0 1]


Structural_Defect       20.08
Surface_Defect           2.68
Contamination_Defect     0.00
dtype: float64

Structural_Defect       1513
Surface_Defect           202
Contamination_Defect       0
dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7535 entries, 0 to 7534
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Surface_Defect        7535 non-null   int64
 1   Structural_Defect     7535 non-null   int64
 2   Contamination_Defect  7535 non-null   int64
dtypes: int64(3)
memory usage: 176.7 KB


# 결측값 처리(Sensor 중심) **!!split 후에 train/test 각각 따로 보간해야 됨!!**
생각해봤는데 이 데이터엔 timestamp가 없고 행 순서가 시간 순서라는 보장도 없는데 선행보간법을 하는게 맞나?\
회의를 한 뒤에 더 안전/간단한 방법인 중앙값으로 파이프라인으로 넣기

_Min/_Max...\
상수값이니까 상관없나

In [10]:
# =========================
# 결측치(NA) 구조 확인 (+ 컬럼별 비율)
# =========================
print("===== 결측치(NA) 확인 =====")

n_rows = len(df)

na_count = df.isna().sum().sort_values(ascending=False)
na_cols = na_count[na_count > 0]

# 그룹별 결측치 개수(총 결측 수) - 기존 그대로
print("그룹별 결측치 합(총 결측 수):")
print(pd.Series({
    "Process": process_df.isna().sum().sum(),
    "Sensor":  sensor_df.isna().sum().sum(),
    "Defects": defects_df.isna().sum().sum()
}))
print()

# 컬럼별 결측치 개수
print("결측치가 있는 컬럼만(개수):")
print(na_cols)
print()

# 컬럼별 결측치 비율(%): (결측 행 수 / 전체 행 수) * 100
print("결측치가 있는 컬럼만(비율 %):")
print(((na_cols / n_rows) * 100).round(4))
print()

===== 결측치(NA) 확인 =====
그룹별 결측치 합(총 결측 수):
Process      0
Sensor     540
Defects      0
dtype: int64

결측치가 있는 컬럼만(개수):
Sensor  Factory_Temp            90
        Factory_Humidity        90
        Factory_Temp_Max        90
        Factory_Humidity_Max    90
        Factory_Humidity_Min    90
        Factory_Temp_Min        90
dtype: int64

결측치가 있는 컬럼만(비율 %):
Sensor  Factory_Temp            1.1944
        Factory_Humidity        1.1944
        Factory_Temp_Max        1.1944
        Factory_Humidity_Max    1.1944
        Factory_Humidity_Min    1.1944
        Factory_Temp_Min        1.1944
dtype: float64



In [11]:
# Sensor에서 _MIN/_MAX로 끝나는 컬럼 찾아서 삭제
minmax_cols = [c for c in sensor_df.columns if c.endswith("_Min") or c.endswith("_Max")]
sensor_df_clean = sensor_df.drop(columns=minmax_cols)

print("삭제한 _MIN/_MAX 컬럼 수:", len(minmax_cols))
print("삭제 전 sensor_df shape:", sensor_df.shape)
print("삭제 후 sensor_df shape:", sensor_df_clean.shape)

삭제한 _MIN/_MAX 컬럼 수: 8
삭제 전 sensor_df shape: (7535, 14)
삭제 후 sensor_df shape: (7535, 6)


In [12]:
# =========================
# 결측치(NA) 구조 확인 (+ 컬럼별 비율)
# =========================
print("===== 결측치(NA) 확인 =====")

n_rows = len(sensor_df_clean)

na_count = sensor_df_clean.isna().sum().sort_values(ascending=False)
na_cols = na_count[na_count > 0]

# 컬럼별 결측치 개수
print("결측치가 있는 컬럼만(개수):")
print(na_cols)
print()

===== 결측치(NA) 확인 =====
결측치가 있는 컬럼만(개수):
Factory_Temp        90
Factory_Humidity    90
dtype: int64



In [15]:
# 중앙값으로 채움
fill_cols = ["Factory_Temp", "Factory_Humidity"]
sensor_df_clean[fill_cols] = sensor_df_clean[fill_cols].fillna(sensor_df_clean[fill_cols].median())

print(sensor_df_clean[fill_cols].isna().sum())
print("남아있는 결측치의 개수:", sensor_df_clean[fill_cols].isna().sum().sum())

Factory_Temp        0
Factory_Humidity    0
dtype: int64
남아있는 결측치의 개수: 0


# 이상치 처리

# 식별자/그룹 구분 
생각해보니까 shot_key 이거 식별자라서 모델링 할땐 무조건 제거해야함.

In [16]:
# 1) shot_key 생성
process_df_clean = process_df.copy()
process_df_clean["shot_key"] = process_df_clean["id"].astype(str) + "_" + process_df_clean["Shot"].astype(str)

# 2) id, Shot 삭제
process_df_clean = process_df_clean.drop(columns=["id", "Shot"])

# 3) shot_key를 맨 앞으로 이동
cols = ["shot_key"] + [c for c in process_df_clean.columns if c != "shot_key"]
process_df_clean = process_df_clean[cols]

# 중복이 있으면 어떤 게 중복인지(있을 때만 의미 있음)
dup = process_df_clean["shot_key"][process_df_clean["shot_key"].duplicated(keep=False)]
print("중복 개수:", dup.shape[0])

중복 개수: 0


In [17]:
print(process_df["Product_Type"].value_counts())

Product_Type
1    4207
2    3328
Name: count, dtype: int64


In [18]:
clean_df = pd.concat([process_df_clean, sensor_df_clean, group_df], axis=1)

print("clean_df shape:", clean_df.shape)
clean_df.head()

clean_df shape: (7535, 25)


,shot_key,Product_Type,Velocity_1,Velocity_2,Velocity_3,High_Velocity,Cylinder_Pressure,Rapid_Rise_Time,Biscuit_Thickness,Clamping_Force,Cycle_Time,Pressure_Rise_Time,Casting_Pressure,Spray_Time,Spray_1_Time,Spray_2_Time,Melting_Furnace_Temp,Air_Pressure,Coolant_Temp,Coolant_Pressure,Factory_Temp,Factory_Humidity,Surface_Defect,Structural_Defect,Contamination_Defect
0,1_1,1,0.144,0.170,0.188,2.134,214,0.008,10,258,20.7,0.044,1037,7.8,0.7,0.8,695.0,6.3,26.0,2.71,32.9,58.4,0,0,0
1,1002_2,1,0.144,0.170,0.182,2.124,217,0.008,11,257,20.7,0.044,1052,7.8,0.7,0.8,696.4,6.3,26.1,2.69,32.9,58.2,0,0,0
2,2003_3,1,0.144,0.170,0.182,2.116,214,0.008,11,257,20.8,0.041,1037,7.8,0.7,0.8,696.4,6.3,26.1,2.69,32.9,58.2,0,0,0
3,3004_4,1,0.144,0.170,0.182,2.137,217,0.008,11,257,20.7,0.043,1051,7.8,0.7,0.8,696.4,6.3,26.1,2.69,32.9,58.2,0,1,0
4,4005_5,1,0.144,0.172,0.176,2.111,217,0.008,12,257,20.7,0.042,1052,7.8,0.7,0.8,697.9,6.4,26.1,2.69,32.9,57.8,0,0,0
